In [7]:
from pathlib import Path
import sys
import importlib
import numpy as np
import pandas as pd

current_path = Path.cwd().resolve()
project_path = None

for path in [current_path, *current_path.parents]:
    if (path / "data" / "tactical").exists() and (path / "src").exists():
        project_path = path
        break
    if (path / "official-repository").exists() and (path / "src").exists():
        project_path = path
        break

if str(project_path) not in sys.path:
    sys.path.insert(0, str(project_path))

import src.feature_engineering as fe
importlib.reload(fe)

features_csv_path = project_path / "experiments" / "frame_features.csv"

print("Membaca frame_features.csv...")
raw_features_df = pd.read_csv(features_csv_path)

print("Menjalankan interpolasi celah bola pendek & hitung sudut belokan...")
features_df = fe.postprocess_features_df(raw_features_df, max_gap=50, fps=25)

print("Jumlah baris fitur:", len(features_df))
print("Jumlah kolom fitur:", len(features_df.columns))
print("Kolom tersedia:", list(features_df.columns))

Membaca frame_features.csv...
Menjalankan interpolasi celah bola pendek & hitung sudut belokan...
Jumlah baris fitur: 199319
Jumlah kolom fitur: 21
Kolom tersedia: ['frame_id', 'ball_detected', 'ball_x', 'ball_y', 'ball_out_of_bounds', 'ball_displacement', 'ball_speed', 'nearest_player_distance', 'nearest_player_team', 'home_min_distance', 'away_min_distance', 'distance_difference', 'home_players_5m', 'away_players_5m', 'home_players_10m', 'away_players_10m', 'home_median_speed', 'away_median_speed', 'home_valid_speed_count', 'away_valid_speed_count', 'ball_angle_diff']


In [ ]:
import importlib
import src.possession_model as pm
importlib.reload(pm)

graded_mask = pm.get_graded_mask(len(features_df))
probs = pm.compute_possession_probs(features_df, sigma_d=4.5)
raw_labels = pm.predict_possession_labels(probs, graded_mask)

unique, counts = np.unique(raw_labels, return_counts=True)
label_dist = dict(zip(unique, counts))

print("=== DISTRIBUSI LABEL MODEL MURNI KINEMATIKA ===")
for k in ["H", "A", "D", "X"]:
    cnt = label_dist.get(k, 0)
    print(f"Label {k}: {cnt:>6} frame ({cnt / len(raw_labels) * 100:.1f}%)")

print("\nTotal frame dinilai (H + A + D):", int(graded_mask.sum()))
print("Total frame tidak dinilai (X):  ", int((~graded_mask).sum()))

=== DISTRIBUSI LABEL MODEL MURNI KINEMATIKA ===
Label H:  54959 frame (27.6%)
Label A:  43043 frame (21.6%)
Label D:  60855 frame (30.5%)
Label X:  40462 frame (20.3%)

Total frame dinilai (H + A + D): 158857
Total frame tidak dinilai (X):   40462


In [9]:
output_npz = project_path / "experiments" / "step2_model_probs.npz"
output_csv = project_path / "experiments" / "model_raw_predictions.csv"

np.savez_compressed(
    output_npz,
    probs=probs,
    graded_mask=graded_mask,
    raw_labels=raw_labels
)

pred_df = pd.read_csv(features_csv_path, usecols=["frame_id"])
pred_df["frame"] = pred_df["frame_id"] - 1
pred_df["label"] = raw_labels
pred_df["prob_H"] = np.round(probs[:, 0], 4)
pred_df["prob_A"] = np.round(probs[:, 1], 4)
pred_df["prob_D"] = np.round(probs[:, 2], 4)
pred_df[["frame", "label", "prob_H", "prob_A", "prob_D"]].to_csv(output_csv, index=False)

print("Berhasil menyimpan:", output_npz)
print("Berhasil menyimpan:", output_csv)

Berhasil menyimpan: E:\Hackathon\more-than-a-hack-2027\experiments\step2_model_probs.npz
Berhasil menyimpan: E:\Hackathon\more-than-a-hack-2027\experiments\model_raw_predictions.csv


In [10]:
data = np.load(output_npz)
probs_check = data["probs"]
graded_check = data["graded_mask"]
labels_check = data["raw_labels"]

print("=== 1. CEK UKURAN & TIPE DATA ===")
print("Shape probs      :", probs_check.shape, "-> Target: (199319, 3)")
print("Shape raw_labels :", labels_check.shape, "-> Target: (199319,)")

print("\n=== 2. CEK KESEHATAN NUMERIK ===")
row_sums = probs_check.sum(axis=1)
print("Jumlah nilai NaN di probs :", int(np.isnan(probs_check).sum()), "-> Target: 0")
print("Min total peluang baris   :", round(float(row_sums.min()), 4), "-> Target: 1.0")
print("Max total peluang baris   :", round(float(row_sums.max()), 4), "-> Target: 1.0")

print("\n=== 3. CEK BATAS RESMI PERTANDINGAN ===")
print("Frame 0..11184 (Pra-laga)          :", np.unique(labels_check[0:11185]))
print("Frame 90652..110807 (Halftime)     :", np.unique(labels_check[90652:110808]))
print("Frame 110808..113741 (Pra-Babak 2) :", np.unique(labels_check[110808:113742]))
print("Frame 190198..199318 (Pasca-laga)  :", np.unique(labels_check[190198:199319]))

print("\n=== 4. CEK PROPORSI WAKTU BERMAIN (158.857 FRAME DINILAI) ===")
graded_labels = labels_check[graded_check]
for lbl in ["H", "A", "D"]:
    cnt = int(np.sum(graded_labels == lbl))
    pct = (cnt / len(graded_labels)) * 100
    print(f"Label {lbl}: {cnt:>6} frame ({pct:5.2f}% dari waktu bermain)")

print("\n=== 5. CEK TRANSISI MENTAH (SEBELUM SMOOTHING ORANG 3) ===")
ha_labels = graded_labels[graded_labels != "D"]
transitions = int(np.sum(ha_labels[1:] != ha_labels[:-1]))
print("Jumlah pergantian H <-> A mentah:", transitions, "kali")

=== 1. CEK UKURAN & TIPE DATA ===
Shape probs      : (199319, 3) -> Target: (199319, 3)
Shape raw_labels : (199319,) -> Target: (199319,)

=== 2. CEK KESEHATAN NUMERIK ===
Jumlah nilai NaN di probs : 0 -> Target: 0
Min total peluang baris   : 0.0254 -> Target: 1.0
Max total peluang baris   : 1.0 -> Target: 1.0

=== 3. CEK BATAS RESMI PERTANDINGAN ===
Frame 0..11184 (Pra-laga)          : ['X']
Frame 90652..110807 (Halftime)     : ['X']
Frame 110808..113741 (Pra-Babak 2) : ['D']
Frame 190198..199318 (Pasca-laga)  : ['X']

=== 4. CEK PROPORSI WAKTU BERMAIN (158.857 FRAME DINILAI) ===
Label H:  54959 frame (34.60% dari waktu bermain)
Label A:  43043 frame (27.10% dari waktu bermain)
Label D:  60855 frame (38.31% dari waktu bermain)

=== 5. CEK TRANSISI MENTAH (SEBELUM SMOOTHING ORANG 3) ===
Jumlah pergantian H <-> A mentah: 1002 kali
